# Logger: PGN + JSONL game records

**Deliverable:** logging. Every game produces

* `<game_id>.jsonl`: one JSON object per ply, written and flushed immediately (a crash mid-game
  keeps everything up to the last move). Fields: FEN before/after, move (UCI + SAN), who moved,
  eval, engine best move, think time, robot attempts, arm plan, free-form extras.
* `<game_id>.pgn`: the game in standard PGN, with per-move comments using the common
  `[%eval ...]` and `[%emt ...]` annotations (readable by lichess / chess GUIs), written at the end.

These files feed the teaching side (`gambit_teach/notebooks/run_analyze_history.ipynb`) and the
README measurement logs (pick-and-place success, retries, game statistics).

**TODO checklist**
- [x] JSONL and PGN writers, reader back to pandas.
- [ ] Decide the player id scheme (name? hashed?) and keep personal data out of shared logs.
- [ ] Log recovery events (kind, action) as separate JSONL lines (`kind: "recovery"`).

In [ ]:
from __future__ import annotations

import datetime as dt
import io
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

import chess
import chess.pgn
import pandas as pd

## Per-move record

`eval_cp` is from **White's** point of view (the PGN convention), after the move.

In [ ]:
@dataclass
class MoveRecord:
    game_id: str
    ply: int
    by: str                               # "human" | "robot"
    fen_before: str
    fen_after: str
    uci: str
    san: str
    think_time_s: float = 0.0
    eval_cp: Optional[int] = None         # White POV, after the move
    best_uci: Optional[str] = None        # engine's best move in fen_before
    robot_attempts: int = 0
    plan: str = ""
    timestamp: str = field(default_factory=lambda: dt.datetime.now().isoformat(timespec="seconds"))
    extra: dict = field(default_factory=dict)
    kind: str = "move"


def format_emt(seconds: float) -> str:
    s = int(round(max(0.0, seconds)))
    return f"{s // 3600}:{(s % 3600) // 60:02d}:{s % 60:02d}"


def move_comment(r: MoveRecord) -> str:
    parts = []
    if r.eval_cp is not None:
        parts.append(f"[%eval {r.eval_cp / 100:.2f}]")
    parts.append(f"[%emt {format_emt(r.think_time_s)}]")
    if r.by == "robot" and r.robot_attempts:
        parts.append(f"robot attempts={r.robot_attempts}")
    if r.best_uci and r.best_uci != r.uci:
        parts.append(f"best={r.best_uci}")
    return " ".join(parts)

## Writer

In [ ]:
class GameLogger:
    def __init__(self, log_dir: Path | str, game_id: Optional[str] = None, headers: Optional[dict] = None,
                 start_fen: str = chess.STARTING_FEN):
        self.dir = Path(log_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.game_id = game_id or dt.datetime.now().strftime("game_%Y%m%d_%H%M%S")
        self.headers = {"Event": "Gambit game", "Site": "Gambit robot",
                        "Date": dt.date.today().strftime("%Y.%m.%d"), **(headers or {})}
        self.start_fen = start_fen
        self.records: list[MoveRecord] = []
        self.jsonl_path = self.dir / f"{self.game_id}.jsonl"
        self.pgn_path = self.dir / f"{self.game_id}.pgn"
        self._fh = open(self.jsonl_path, "a")

    def log(self, record: MoveRecord) -> None:
        self.records.append(record)
        self._fh.write(json.dumps(asdict(record)) + "\n")
        self._fh.flush()

    def log_event(self, kind: str, **data) -> None:
        """Non-move lines (recovery, errors, notes) in the same JSONL file."""
        self._fh.write(json.dumps({"game_id": self.game_id, "kind": kind,
                                   "timestamp": dt.datetime.now().isoformat(timespec="seconds"), **data}) + "\n")
        self._fh.flush()

    def to_pgn(self, result: str = "*") -> chess.pgn.Game:
        game = chess.pgn.Game()
        for k, v in self.headers.items():
            game.headers[k] = str(v)
        if self.start_fen != chess.STARTING_FEN:
            game.setup(chess.Board(self.start_fen))
        game.headers["Result"] = result
        node = game
        for r in self.records:
            node = node.add_variation(chess.Move.from_uci(r.uci))
            node.comment = move_comment(r)
        return game

    def finish(self, result: str = "*") -> Path:
        self.pgn_path.write_text(str(self.to_pgn(result)) + "\n\n")
        self.close()
        return self.pgn_path

    def close(self) -> None:
        if not self._fh.closed:
            self._fh.close()

## Readers

In [ ]:
def read_jsonl(path: Path | str, kind: Optional[str] = "move") -> pd.DataFrame:
    rows = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    if kind is not None and "kind" in df:
        df = df[df["kind"] == kind].reset_index(drop=True)
    return df


def read_pgn_games(path: Path | str) -> list[chess.pgn.Game]:
    games = []
    with open(path) as f:
        while (g := chess.pgn.read_game(f)) is not None:
            games.append(g)
    return games

## Self-check (writes only to a temporary folder)

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    lg = GameLogger(tmp, "test_game", {"White": "human", "Black": "Gambit"})
    b = chess.Board()
    for i, san in enumerate(["e4", "e5", "Nf3"]):
        mv = b.parse_san(san); before = b.fen(); b.push(mv)
        lg.log(MoveRecord("test_game", i, "human" if i % 2 == 0 else "robot", before, b.fen(), mv.uci(), san,
                          think_time_s=3.4 * (i + 1), eval_cp=20 + i, robot_attempts=i % 2))
    lg.log_event("recovery", mismatch="misplaced", action="arm_retry")
    pgn_path = lg.finish("*")
    df = read_jsonl(lg.jsonl_path)
    assert len(df) == 3 and list(df.san) == ["e4", "e5", "Nf3"]
    assert len(read_jsonl(lg.jsonl_path, kind=None)) == 4
    g, = read_pgn_games(pgn_path)
    assert [m.uci() for m in g.mainline_moves()] == ["e2e4", "e7e5", "g1f3"]
    first = next(iter(g.mainline()))
    assert first.eval().white().score() == 20 and abs(first.emt() - 3) < 1e-6, first.comment
    assert "robot attempts=1" in first.next().comment
assert format_emt(3725) == "1:02:05"
print("logger self-check passed")